<a href="https://colab.research.google.com/github/lilian662/SIMULACION-I/blob/main/EJERCICIOSTERCERPARCIAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Tercer parcial**


## **Edith Lilian Castillo Ojeda**

**21/06/26**

**Simulación I**

# Simulación de una línea de espera con dos servidores en serie.

## Variables del modelo

### Variable de tiempo

`t`: representa el tiempo actual de la simulación.

### Variable de estado

Ross define el estado como:

$$
ES=(n_1,n_2)
$$

donde:

- `n1` es el número de clientes en el servidor 1, incluyendo los que esperan y el que está en servicio.
- `n2` es el número de clientes en el servidor 2, incluyendo los que esperan y el que está en servicio.


### Variables de conteo

`NA`: número de llegadas hasta el instante `t`.

`ND`: número de salidas del sistema hasta el instante `t`.


### Variables de salida

`A1(n)`: hora de llegada del cliente `n` al sistema.

`A2(n)`: hora de llegada del cliente `n` al servidor 2.

`D(n)`: hora de salida del cliente `n` del sistema.
## Lista de eventos

La lista de eventos es:

$$
LE=(t_A,t_1,t_2)
$$

donde:

- `tA` es la hora de la siguiente llegada.
- `t1` es la hora en que concluye el servicio del cliente atendido por el servidor 1.
- `t2` es la hora en que concluye el servicio del cliente atendido por el servidor 2.

Si en algún momento no hay cliente siendo atendido por un servidor, su tiempo correspondiente se toma como infinito.

$$
t_i=\infty
$$



## Ejemplo utilizado

Se propone el siguiente ejemplo para ilustrar el funcionamiento del algoritmo:


Se considera un proceso Poisson no homogéneo con intensidad

$
\lambda(t)=2+\sin(t)
$

Esta función cumple:

$
1 \le \lambda(t) \le 3
$

por lo que se utiliza

$
\lambda^*=3
$

como cota superior en el algoritmo de aceptación-rechazo propuesto por Ross para generar las llegadas.


### Tiempo de servicio del servidor 1

Se supone una distribución exponencial:

$
G_1 \sim Exp(4)
$

con media

$
E[G_1]=\frac{1}{4}=0.25
$

unidades de tiempo.


### Tiempo de servicio del servidor 2

Se supone una distribución exponencial:

$
G_2 \sim Exp(5)
$

con media

$
E[G_2]=\frac{1}{5}=0.20
$

unidades de tiempo.



## Resultados

Al finalizar la simulación se construye una tabla con la información de cada cliente:

- `Cliente`: número del cliente.
- `A1(n)`: hora en que el cliente llega al sistema.
- `A2(n)`: hora en que el cliente llega al servidor 2.
- `D(n)`: hora en que el cliente sale del sistema.

A partir de estos datos se calculan:

### Tiempo en el servidor 1

$$
T_1=A_2(n)-A_1(n)
$$

### Tiempo en el servidor 2

$$
T_2=D(n)-A_2(n)
$$

### Tiempo total en el sistema

$$
T=D(n)-A_1(n)
$$

Finalmente, el programa muestra los promedios de tiempo en el servidor 1, en el servidor 2 y en todo el sistema.


In [2]:
import numpy as np
import pandas as pd
import math

np.random.seed(123)

# ejemplo


def lam(t):
    return 2 + np.sin(t)

lambda_max = 3

def generar_Ts(s):
    t = s
    while True:
        U = np.random.rand()
        t = t - (1 / lambda_max) * np.log(U)

        U = np.random.rand()
        if U <= lam(t) / lambda_max:
            return t

def G1():
    return np.random.exponential(scale=1/4)

def G2():
    return np.random.exponential(scale=1/5)


# inicio


N = 100

t = 0
NA = 0
ND = 0

ES = (0, 0)
n1, n2 = ES

T0 = generar_Ts(0)

tA = T0
t1 = math.inf
t2 = math.inf

A1 = {}
A2 = {}
D = {}

clients_entering_s2 = 0


# algoritmo


while ND < N:

    # Lista de eventos
    LE = (tA, t1, t2)


    # CASO 1: tA = min(tA, t1, t2)


    if tA == min(LE):

        t = tA

        NA = NA + 1
        n1 = n1 + 1

        Tr = generar_Ts(t)
        tA = Tr

        if n1 == 1:
            Y1 = G1()
            t1 = t + Y1

        A1[NA] = t


    # CASO 2:  t1 < tA, t1 <= t2

    elif t1 < tA and t1 <= t2:

        t = t1

        n1 = n1 - 1
        n2 = n2 + 1

        if n1 == 0:
            t1 = math.inf
        else:
            Y1 = G1()
            t1 = t + Y1

        if n2 == 1:
            Y2 = G2()
            t2 = t + Y2


        clients_entering_s2 += 1
        A2[clients_entering_s2] = t


    # CASO 3 t2 < tA, t2 < t1
    elif t2 < tA and t2 < t1:

        t = t2

        ND = ND + 1
        n2 = n2 - 1

        if n2 == 0:
            t2 = math.inf
        else:
            Y2 = G2()
            t2 = t + Y2

        D[ND] = t


# resultados en la tabla


tabla = pd.DataFrame({
    "Cliente": range(1, N + 1),
    "A1(n) llegada al sistema": [A1[i] for i in range(1, N + 1)],
    "A2(n) llegada servidor 2": [A2[i] for i in range(1, N + 1)],
    "D(n) salida del sistema": [D[i] for i in range(1, N + 1)]
})

tabla["Tiempo servidor 1"] = tabla["A2(n) llegada servidor 2"] - tabla["A1(n) llegada al sistema"]
tabla["Tiempo servidor 2"] = tabla["D(n) salida del sistema"] - tabla["A2(n) llegada servidor 2"]
tabla["Tiempo total"] = tabla["D(n) salida del sistema"] - tabla["A1(n) llegada al sistema"]

display(tabla)

print("Promedio tiempo servidor 1:", tabla["Tiempo servidor 1"].mean())
print("Promedio tiempo servidor 2:", tabla["Tiempo servidor 2"].mean())
print("Promedio tiempo total en el sistema:", tabla["Tiempo total"].mean())
print("Total de llegadas NA:", NA)
print("Total de salidas ND:", ND)

,Cliente,A1(n) llegada al sistema,A2(n) llegada servidor 2,D(n) salida del sistema,Tiempo servidor 1,Tiempo servidor 2,Tiempo total
0,1,0.120577,0.438345,0.548364,0.317768,0.110020,0.427787
1,2,0.615064,0.778994,0.894448,0.163930,0.115454,0.279384
2,3,0.621538,1.105449,1.145748,0.483911,0.040299,0.524209
3,4,0.933603,1.440297,1.478881,0.506694,0.038584,0.545278
4,5,1.873201,2.124755,2.381101,0.251555,0.256346,0.507901
...,...,...,...,...,...,...,...
95,96,58.476323,58.626278,58.693961,0.149954,0.067684,0.217638
96,97,59.462463,59.699266,60.127892,0.236803,0.428626,0.665429
97,98,59.532492,60.684852,60.801289,1.152360,0.116437,1.268797
98,99,60.037900,60.910865,61.012220,0.872964,0.101355,0.974319


Promedio tiempo servidor 1: 0.36242889286237523
Promedio tiempo servidor 2: 0.4054046766790986
Promedio tiempo total en el sistema: 0.767833569541474
Total de llegadas NA: 101
Total de salidas ND: 100


## Conclusión

Se implementó una simulación para una línea de espera con dos servidores en serie siguiendo el pseudocódigo propuesto por Ross. La simulación permitió registrar las llegadas, los cambios entre servidores y las salidas del sistema. A partir de los datos obtenidos fue posible calcular los tiempos de permanencia en cada servidor y el tiempo total dentro del sistema. Este tipo de modelos resulta útil para analizar procesos secuenciales donde cada cliente debe pasar por varias etapas de servicio.

# Sistema de línea de espera con dos servidores en paralelo

En este sistema los clientes llegan a una sola fila y pueden ser atendidos por cualquiera de los dos servidores.


## Diferencia con el sistema en serie

En el sistema en paralelo se tiene:

Llegada → Servidor 1 → Salida  
Llegada → Servidor 2 → Salida

Es decir, cada cliente solo necesita ser atendido por uno de los dos servidores.


## Variables del modelo

### Variable de tiempo

`t`: representa el tiempo actual de la simulación.

### Variables de conteo

`NA`: número de llegadas hasta el instante `t`.

`C1`: número de clientes atendidos por el servidor 1.

`C2`: número de clientes atendidos por el servidor 2.

### Variable de estado

Ross define el estado como:

$$
ES=(n,i_1,i_2,\ldots,i_n)
$$

donde:

- `n` es el número de clientes en el sistema.
- `i1` es el cliente que está siendo atendido por el servidor 1.
- `i2` es el cliente que está siendo atendido por el servidor 2.
- Los demás clientes están en fila.

En el código se representa de forma equivalente usando:

- `servidor1`
- `servidor2`
- `cola`


## Lista de eventos

La lista de eventos es:

$$
LE=(t_A,t_1,t_2)
$$

donde:

- `tA` es la hora de la siguiente llegada.
- `t1` es la hora en que termina el servicio del cliente atendido por el servidor 1.
- `t2` es la hora en que termina el servicio del cliente atendido por el servidor 2.

En cada iteración se selecciona el menor de estos tiempos.


## Ejemplo propuesto

Se considera un proceso Poisson no homogéneo con intensidad:
(utilizamos el mismo ejemplo que para los servidores en serie)
$$
\lambda(t)=2+\sin(t)
$$

Como:

$$
1 \leq \lambda(t) \leq 3
$$

se usa:

$$
\lambda^*=3
$$

como cota superior para generar las llegadas mediante el método de aceptación-rechazo de Ross.

Para los tiempos de servicio se supone:

$$
G_1 \sim Exp(4)
$$

y

$$
G_2 \sim Exp(5)
$$

Por lo tanto, las medias de servicio son:

$$
E[G_1]=\frac{1}{4}=0.25
$$

$$
E[G_2]=\frac{1}{5}=0.20
$$


## Resultados

Se registran:

- `A(n)`: hora de llegada del cliente n.
- `D(n)`: hora de salida del cliente n.
- Servidor que atendió al cliente.
- Tiempo total en el sistema.

El tiempo total se calcula como:

$$
D(n)-A(n)
$$



In [3]:
import numpy as np
import pandas as pd
import math

np.random.seed(123)


# ejemplo


def lam(t):
    return 2 + np.sin(t)

lambda_max = 3

def generar_Ts(s):
    t = s
    while True:
        U = np.random.rand()
        t = t - (1 / lambda_max) * np.log(U)

        U = np.random.rand()
        if U <= lam(t) / lambda_max:
            return t

def G1():
    return np.random.exponential(scale=1/4)

def G2():
    return np.random.exponential(scale=1/5)


# Inicio


N = 100

t = 0
NA = 0
C1 = 0
C2 = 0

servidor1 = None
servidor2 = None
cola = []

tA = generar_Ts(0)
t1 = math.inf
t2 = math.inf

A = {}
D = {}
servidor_cliente = {}


# algoritmo


while len(D) < N:

    LE = (tA, t1, t2)


    # CASO 1: LLEGADA: tA = min(tA, t1, t2)

    if tA == min(LE):

        t = tA
        NA = NA + 1

        A[NA] = t

        Tr = generar_Ts(t)
        tA = Tr

        # Si el sistema está vacío
        if servidor1 is None and servidor2 is None and len(cola) == 0:

            servidor1 = NA
            Y1 = G1()
            t1 = t + Y1

        # Si servidor 1 ocupado y servidor 2 libre
        elif servidor1 is not None and servidor2 is None and len(cola) == 0:

            servidor2 = NA
            Y2 = G2()
            t2 = t + Y2

        # Si servidor 1 libre y servidor 2 ocupado
        elif servidor1 is None and servidor2 is not None and len(cola) == 0:

            servidor1 = NA
            Y1 = G1()
            t1 = t + Y1

        # Si ambos servidores están ocupados
        else:

            cola.append(NA)


    # CASO 2: SALE SERVIDOR 1: t1 < tA, t1 <= t2
    elif t1 < tA and t1 <= t2:

        t = t1

        cliente_sale = servidor1
        C1 = C1 + 1

        D[cliente_sale] = t
        servidor_cliente[cliente_sale] = 1

        # Si no hay cola, el servidor 1 queda libre
        if len(cola) == 0:

            servidor1 = None
            t1 = math.inf

        # Si hay cola, entra el primero a servidor 1
        else:

            servidor1 = cola.pop(0)
            Y1 = G1()
            t1 = t + Y1


    # CASO 3: SALE SERVIDOR 2: t2 < tA, t2 < t1
    #Nota: El libro deja las actualizaciones del Caso 3 como ejercicio; en la implementación se utilizó la misma lógica del Caso 2 aplicada al servidor 2.

    elif t2 < tA and t2 < t1:

        t = t2

        cliente_sale = servidor2
        C2 = C2 + 1

        D[cliente_sale] = t
        servidor_cliente[cliente_sale] = 2

        # Si no hay cola, el servidor 2 queda libre
        if len(cola) == 0:

            servidor2 = None
            t2 = math.inf

        # Si hay cola, entra el primero a servidor 2
        else:

            servidor2 = cola.pop(0)
            Y2 = G2()
            t2 = t + Y2

# resultados


tabla = pd.DataFrame({
    "Cliente": range(1, N + 1),
    "A(n) llegada": [A[i] for i in range(1, N + 1)],
    "D(n) salida": [D[i] for i in range(1, N + 1)],
    "Servidor": [servidor_cliente[i] for i in range(1, N + 1)]
})

tabla["Tiempo total en sistema"] = tabla["D(n) salida"] - tabla["A(n) llegada"]

display(tabla)

print("Clientes atendidos por servidor 1:", C1)
print("Clientes atendidos por servidor 2:", C2)
print("Total de llegadas NA:", NA)
print("Total de salidas:", len(D))
print("Promedio de tiempo en el sistema:", tabla["Tiempo total en sistema"].mean())
print("Tiempo final de simulación:", t)

,Cliente,A(n) llegada,D(n) salida,Servidor,Tiempo total en sistema
0,1,0.120577,0.438345,1,0.317768
1,2,0.615064,0.739507,1,0.124443
2,3,1.027969,1.172287,1,0.144318
3,4,1.384471,1.719320,1,0.334848
4,5,2.324069,2.513651,1,0.189582
...,...,...,...,...,...
95,96,54.158028,54.772538,2,0.614510
96,97,54.892540,55.051585,1,0.159045
97,98,55.156625,55.263091,1,0.106466
98,99,56.931078,57.009816,1,0.078738


Clientes atendidos por servidor 1: 69
Clientes atendidos por servidor 2: 31
Total de llegadas NA: 100
Total de salidas: 100
Promedio de tiempo en el sistema: 0.22663915667422838
Tiempo final de simulación: 57.059518380279194


## Conclusión

Se implementó una simulación para un sistema con dos servidores en paralelo utilizando la metodología propuesta por Ross. El modelo permitió analizar la asignación de clientes a los servidores y el comportamiento de la fila de espera. Además, se obtuvieron tiempos de permanencia y estadísticas de servicio que permiten evaluar el desempeño del sistema. Este tipo de estructuras es común en centros de atención donde varios servidores pueden atender clientes de manera simultánea.

# Variables Antitéticas






Las variables antitéticas son una técnica de reducción de varianza utilizada en simulación Monte Carlo.

Partimos de la expresión

$$
Var\left(\frac{X_1+X_2}{2}\right)
=
\frac{1}{4}
\left[
Var(X_1)+Var(X_2)+2Cov(X_1,X_2)
\right]
$$

donde

$$
Cov(X_1,X_2)
=
E[X_1X_2]-E[X_1]E[X_2].
$$

La expresión anterior muestra que la varianza puede reducirse si la covarianza entre $X_1$ y $X_2$ es negativa.

Por esta razón se utilizan variables antitéticas, es decir, variables aleatorias con correlación negativa.

Un ejemplo clásico es:

$$
U \sim U(0,1)
$$

y su variable antitética

$$
1-U.
$$

Si $U$ toma valores grandes, entonces $1-U$ toma valores pequeños y viceversa.

En este trabajo se desea estimar la integral

$$
\theta=
\int_0^1 e^{x^2}\,dx.
$$

Definiendo

$$
f(x)=e^{x^2},
$$

el estimador de Monte Carlo crudo es

$$
X_1=
\frac{1}{N}
\sum_{i=1}^{N}
f(U_i),
\qquad U_i\sim U(0,1).
$$

Utilizando variables antitéticas se construye además

$$
X_2=
\frac{1}{N}
\sum_{i=1}^{N}
f(1-U_i).
$$

Finalmente, el estimador antitético se obtiene promediando ambos estimadores:

$$
X_A=
\frac{X_1+X_2}{2}.
$$

La eficiencia del método se evalúa comparando la varianza obtenida mediante Monte Carlo crudo con la varianza obtenida usando variables antitéticas.

### EJEMPLO:




 Variables antitéticas aplicadas a la estimación de la integral de 0 a 1 de exp(x2).

In [4]:
import numpy as np
import random as r

# Función a integrar
def f(x):
    return np.exp(x**2)

# Monte Carlo crudo
def mc_crudo(f,N):
    s = 0
    for i in range(N):
        U = r.random()
        s += f(U)
    return s/N

# Monte Carlo con variables antitéticas
def mc_antitetico(f,N):
    s = 0
    for i in range(N):
        U = r.random()
        s += (f(U) + f(1-U))/2
    return s/N

N = 1000      # tamaño de muestra
R = 100       # número de repeticiones

L_crudo = []
L_ant = []

for i in range(R):
    M1 = mc_crudo(f,N)
    M2 = mc_antitetico(f,N)

    L_crudo.append(M1)
    L_ant.append(M2)

media_crudo = np.mean(L_crudo)
var_crudo = np.var(L_crudo)

media_ant = np.mean(L_ant)
var_ant = np.var(L_ant)

reduccion = (var_crudo - var_ant)/var_crudo

print("Valor aproximado de la integral")
print("Monte Carlo crudo =", media_crudo)
print("Variables antiteticas =", media_ant)

print("\nVarianzas")
print("Varianza MC crudo =", var_crudo)
print("Varianza antitetica =", var_ant)

print("\nReduccion de varianza")
print(reduccion)
print(reduccion*100,"%")

Valor aproximado de la integral
Monte Carlo crudo = 1.4613616276638288
Variables antiteticas = 1.4626098312000229

Varianzas
Varianza MC crudo = 0.00024427672519200056
Varianza antitetica = 3.15426859732422e-05

Reduccion de varianza
0.8708731421364447
87.08731421364448 %


# Técnicas de remuestreo: Bootstrap y Jackknife.

# Método Bootstrap

El método Bootstrap es una técnica de remuestreo propuesta por Efron que permite estimar la distribución de un estadístico utilizando únicamente la muestra observada.

Sea una muestra aleatoria

$$
X_1,X_2,\ldots,X_n
$$

proveniente de una distribución desconocida \(F\).

La distribución \(F\) se aproxima mediante la distribución empírica


$$
F_e(x)=\frac{\#\{i:X_i\leq x\}}{n}
$$

donde $(\#\{i:X_i\leq x\})$ representa el número de observaciones de la muestra que son menores o iguales que \(x\), y \(n\) es el tamaño de la muestra.

A partir de esta distribución empírica se generan muestras Bootstrap mediante muestreo con reemplazo.

Para cada muestra Bootstrap se calcula el estadístico de interés. En este ejercicio se utiliza la media muestral.

Finalmente se obtiene:

- La media de las medias Bootstrap.
- La desviación estándar Bootstrap.
- Un intervalo de confianza del 95% utilizando los percentiles 2.5% y 97.5%.

###**EJEMPLO:** Bootstrap

{35, 42, 38, 40, 45, 37, 39, 41, 44, 36, 43, 40}

In [5]:
import numpy as np

datos = np.array([35,42,38,40,45,37,39,41,44,36,43,40])

B = 1000
n = len(datos)

medias_bootstrap = []

for i in range(B):

    muestra_boot = np.random.choice(datos,
                                    size=n,
                                    replace=True)

    medias_bootstrap.append(np.mean(muestra_boot))

media_boot = np.mean(medias_bootstrap)
desv_boot = np.std(medias_bootstrap)

ic_inf = np.percentile(medias_bootstrap,2.5)
ic_sup = np.percentile(medias_bootstrap,97.5)

print("Media Bootstrap =",media_boot)
print("Desviacion estandar =",desv_boot)
print("IC 95% = [",ic_inf,",",ic_sup,"]")

Media Bootstrap = 40.003416666666666
Desviacion estandar = 0.8534542516867686
IC 95% = [ 38.416666666666664 , 41.668749999999996 ]


# Método Jackknife

El método Jackknife es una técnica de remuestreo que consiste en eliminar una observación de la muestra original en cada iteración.

Si la muestra original contiene \(n\) observaciones, se generan \(n\) muestras de tamaño \(n-1\).

Para cada muestra se calcula el estadístico de interés

$$
\hat{\theta}_i
$$

y posteriormente se obtiene el estimador Jackknife

$$
\hat{\theta}
=
\frac{1}{n}
\sum_{i=1}^{n}
\hat{\theta}_i
$$

La varianza Jackknife se calcula mediante

$$
Var_{jack}
=
\frac{n-1}{n}
\sum_{i=1}^{n}
(\hat{\theta}_i-\hat{\theta})^2
$$

En este ejercicio el estadístico utilizado es la media muestral.

###**EJEMPLO:** Jackknife

{35, 42, 38, 40, 45, 37, 39, 41, 44, 36, 43, 40}

In [6]:
import numpy as np

datos = np.array([35,42,38,40,45,37,39,41,44,36,43,40])

n = len(datos)

medias_jack = []

for i in range(n):

    muestra = np.delete(datos,i)

    medias_jack.append(np.mean(muestra))

theta_jack = np.mean(medias_jack)

var_jack = ((n-1)/n) * np.sum(
    (np.array(medias_jack)-theta_jack)**2
)

print("Media Jackknife =",theta_jack)
print("Varianza Jackknife =",var_jack)
print("Desviacion estandar =",np.sqrt(var_jack))

Media Jackknife = 40.0
Varianza Jackknife = 0.8333333333333355
Desviacion estandar = 0.912870929175278


Los métodos Bootstrap y Jackknife fueron aplicados a la muestra de 12 observaciones proporcionada.

Mediante Bootstrap se generaron 1000 muestras con reemplazo y se calculó la media de cada una. Se obtuvo una media Bootstrap de 40.0034, una desviación estándar de 0.8535 y un intervalo de confianza del 95% igual a [38.42, 41.67].

Posteriormente se aplicó el método Jackknife, generando 12 muestras de tamaño 11 al eliminar una observación distinta en cada iteración. La media Jackknife obtenida fue de 40.0 y la desviación estándar de 0.9129.

Los resultados muestran que ambos métodos producen estimaciones muy cercanas a la media muestral original, igual a 40, lo que indica estabilidad en la estimación del parámetro de interés. Además, las medidas de variabilidad obtenidas por Bootstrap y Jackknife son consistentes entre sí.